# Agentic RAG：誰決定要查？

同一個 agent 先接「你好」，再接「幾點打烊」。預測哪一題需要查資料庫，哪一題第一次會找不到。

這是課程新寫的可重現實驗；資料為虛構餐廳示例。預設使用真正的 LangGraph、稀疏向量索引和可檢查的固定策略，不使用模型或真實搜尋服務。第一格需網路安裝套件，之後執行不需要金鑰。

## 1. 準備環境
Colab：選「執行階段 → 全部執行」。本機：依 README 建立 Python 3.12 環境。

In [ ]:
# Colab 第一次執行會安裝課程固定版本；本機已按 README 安裝時不連網。
from importlib.metadata import version, PackageNotFoundError
import sys, subprocess
try:
    installed = version("langgraph")
except PackageNotFoundError:
    installed = None
if installed != "1.2.20":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.20"])
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add
from math import sqrt

## 2. 檢查可以引用的資料
只有本地菜單、營業時間與一筆外部預報樣本。先找出你預期回答會引用的來源 ID。

In [ ]:
# 小型稀疏向量索引：維度是詞彙；向量餘弦相似度決定排序，不呼叫 embedding API。
LOCAL = [
    {"id":"menu", "kind":"menu", "tags":["菜單","素食","餐點"], "text":"素食菜單有絲瓜蒸餃。"},
    {"id":"hours", "kind":"hours", "tags":["營業","時間"], "text":"教學餐廳的營業時間為 11:00–20:00。"},
]
WEB = [{"id":"web-demo", "kind":"weather", "tags":["天氣","下雨"], "text":"課堂固定預報樣本：有雨。"}]
REFUSAL = "目前沒有足夠來源回答。"
VOCAB = sorted({tag for doc in LOCAL for tag in doc["tags"]})
def vector(query):
    return [float(tag in query) for tag in VOCAB]
def cosine(x, y):
    return sum(a*b for a,b in zip(x,y)) / (sqrt(sum(a*a for a in x)*sum(b*b for b in y)) or 1)
def search_local(query):
    scored = [(cosine(vector(query),vector(" ".join(d["tags"]))),d) for d in LOCAL]
    score, doc = max(scored, key=lambda pair: pair[0])
    return [doc] if score > 0 else []
def intent(question):
    for doc in LOCAL + WEB:
        if any(tag in question for tag in doc["tags"]):
            return doc["kind"]
    return "hours" if "打烊" in question else "unknown"
def rewritten(question):
    return {"hours":"營業時間","menu":"素食菜單","weather":"天氣樣本"}.get(intent(question), question)
def cite(doc):
    return doc["text"] + " [" + doc["id"] + "]"

class State(TypedDict, total=False):
    question: str
    query: str
    docs: list
    answer: str
    route: str
    relevant: bool
    verdict: str
    retries: int
    generations: int
    fault: str
    trace: Annotated[list[str], add]

def step(name, **values):
    return {"trace":[name], **values}
def retrieve(s):
    return step("Retrieve", docs=search_local(s["query"]))
def grade(s):
    ok = any(d["kind"] == intent(s["question"]) for d in s["docs"])
    return step("grade_documents", relevant=ok)
def transform(s):
    return step("transform_query", query=rewritten(s["question"]), retries=s.get("retries",0)+1)
def generate(s):
    doc = next((d for d in s["docs"] if d["kind"]==intent(s["question"])), None)
    return step("Generate", answer=cite(doc) if doc else REFUSAL, generations=s.get("generations",0)+1)
def abstain(s):
    return step("Abstain", answer=REFUSAL)
def web_fixture(s):
    # 固定外部結果，不會真的搜尋網路；未知主題一律回空。
    return step("Web Search (fixture)", docs=[d for d in WEB if any(t in s["query"] for t in d["tags"])])

def run(question, **kw):
    out=graph.invoke({"question":question,"query":question,**kw},{"recursion_limit":30})
    print(" → ".join(out["trace"]), "\n", out["answer"])
    return out

## 3. 把投影片的方塊和箭頭接成圖
每個節點回傳部分 state；每條條件邊決定下一步。追 trace 觀察實際走法。

In [ ]:
def agent(s):
    # 固定策略可重現分支。真實 SDK 版會由 Claude 決定是否呼叫檢索工具，見 README。
    direct = s["question"].strip() in {"嗨", "你好"}
    return step("Agent", route="direct" if direct else "retrieve")
def direct(s):
    return step("Direct",answer="你好！你想查菜單或營業時間嗎？")

g=StateGraph(State)
for name, fn in {"agent":agent, "retrieve":retrieve, "grade":grade, "rewrite":transform,
                 "generate":generate, "direct":direct, "stop":abstain}.items():
    g.add_node(name,fn)
g.add_edge(START,"agent")
g.add_conditional_edges("agent",lambda s:s["route"],["retrieve","direct"])
g.add_edge("retrieve","grade")
g.add_conditional_edges("grade",lambda s:"generate" if s["relevant"] else
                        ("rewrite" if s.get("retries",0)<1 else "stop"),["generate","rewrite","stop"])
g.add_edge("rewrite","agent")
for name in ("direct","generate","stop"): g.add_edge(name,END)
graph=g.compile()

## 4. 執行驗收
斷言失敗表示實際分支或答案與規格不符。

In [ ]:
assert run("你好")["trace"] == ["Agent","Direct"]
assert run("有哪些素食菜單？")["answer"] == cite(LOCAL[0])
x=run("幾點打烊？")
assert x["trace"]==["Agent","Retrieve","grade_documents","transform_query","Agent","Retrieve","grade_documents","Generate"]
assert x["answer"]==cite(LOCAL[1])
assert run("停車費多少？")["answer"]==REFUSAL

## 5. 自己動手
在 LOCAL 新增一份停車文件與標籤，重跑資料和 graph 兩格。哪一條 trace 從拒答變成有來源的答案？

下一步：需要測真實模型時，依資料夾 README 的「Claude Agent SDK」執行獨立的 live graph；不會影響這本的無金鑰驗收。